# Distributed Inference with Gemma 3 on Kaggle TPU v5e-8

This notebook demonstrates how to modernize your JAX/TPU workflows using **Gemma 3** and the **Keras 3 Distribution API**.

By leveraging a Kaggle TPU v5e-8 (8-core mesh), we can perform true Data Parallelism for high-throughput batch inference across a 32k context window without relying on legacy Flax abstractions.

In [ ]:
!pip install -U keras keras-nlp
!pip install -U jax jaxlib

## 1. Environment Setup

Set the backend to `jax` and configure Keras to allocate memory across the TPU mesh.

In [ ]:
import os

# Must be set before importing Keras
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.9"

## 2. Initialize Keras 3 Distribution API

We use `keras.distribution.DataParallel` to automatically shard our input data across all 8 TPU cores. Keras will replicate the model weights on each device and independently process micro-batches.

In [ ]:
import keras
import keras_nlp
import jax

print(f"Devices available: {jax.devices()}")

# Create an 8-core data parallel distribution mesh
devices = keras.distribution.list_devices()
data_parallel = keras.distribution.DataParallel(devices=devices)

# Set the global distribution config
keras.distribution.set_distribution(data_parallel)
print("Distribution API configured!")

## 3. Load Gemma 3 Model

Because the global distribution is set to `DataParallel`, when we instantiate `Gemma3CausalLM`, Keras automatically replicates the weights across all 8 devices.

In [ ]:
# Load Gemma 3 (e.g., 4b parameter version)
model_id = "gemma3_4b_en" # Replace with standard Kaggle preset
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset(model_id)

gemma_lm.summary()

## 4. Run Data-Parallel Batch Inference

When passing a list of prompts, the workload is automatically sharded across the TPU cores.

In [ ]:
batch_prompts = [
    "Explain the significance of true data parallelism in deep learning.",
    "What are the key architectural improvements in Gemma 3?",
    "Write a Python script to calculate Fibonacci using dynamic programming.",
    "Describe the benefits of using JAX over PyTorch for TPU hardware.",
    "How does the Keras 3 Distribution API simplify multi-core scaling?",
    "Generate a summary of global warming mitigation strategies.",
    "Write a creative short story about a sentient robot exploring Mars.",
    "What is the maximum context window supported by Gemma 3, and how is it achieved?"
]

# Inference executes in parallel across the v5e-8 mesh
responses = gemma_lm.generate(batch_prompts, max_length=512)

for i, response in enumerate(responses):
    print(f"\n--- Prompt {i+1} ---\n{response}")